In [1]:
import pandas as pd
import numpy as np

In [2]:
# 1. LOAD RAW DATASETS

# Ingest raw operational data for profiling
# Note: Variable names are maintained, though file paths suggest a naming mismatch in the raw data
df_shipments = pd.read_csv('../data/01_raw/shipments.csv')
df_mapping = pd.read_csv('../data/01_raw/factory_coordinates.csv')
df_coords = pd.read_csv('../data/01_raw/product_factory_mapping.csv')

In [3]:
# 2. DEFINING THE PROFILING ENGINE
def profile_dataframe(df, dataset_name):
    """
    Executes Levels 1 through 4 of the Validation Hierarchy.
    Generates a comprehensive diagnostic report for a single dataframe.
    """
    print(f"======================================================")
    print(f"📊 DATASET PROFILE: {dataset_name.upper()}")
    print(f"======================================================")

    # Level 1: Ingestion & Shape
    print(f"\n[LEVEL 1] SHAPE & SIZE:")
    print(f"Rows: {df.shape[0]:,}")
    print(f"Columns: {df.shape[1]}")

    # Level 2 & 3: Schema, Types, and Completeness
    print(f"\n[LEVEL 2 & 3] SCHEMA & COMPLETENESS:")
    profile_df = pd.DataFrame({
        'Data Type': df.dtypes,
        'Total Nulls': df.isnull().sum(),
        '% Null': (df.isnull().sum() / len(df) * 100).round(2),
        'Unique Values': df.nunique()
    })
    print(profile_df)

    # Level 4: Uniqueness
    print(f"\n[LEVEL 4] UNIQUENESS:")
    duplicate_rows = df.duplicated().sum()
    print(f"Fully Duplicated Rows: {duplicate_rows}")
    print("\n")

# Execute baseline profiling across all staging datasets
profile_dataframe(df_shipments, "Main Shipments")
profile_dataframe(df_mapping, "Product -> Factory Mapping")
profile_dataframe(df_coords, "Factory Coordinates")


# Validate numerical domain constraints (e.g., negative cost/sales anomalies)
print("======================================================")
print("🔍 [LEVEL 5] BUSINESS DOMAIN SANITY CHECKS")
print("======================================================")
print("\nShipments - Numerical Summary:")
print(df_shipments[['Sales', 'Units', 'Gross Profit', 'Cost']].describe().T)

# Validate geospatial coordinate boundaries
print("\nFactory Coordinates - Geospatial Bounds:")
print(f"Latitude Min/Max: {df_coords['Latitude'].min()} / {df_coords['Latitude'].max()}")
print(f"Longitude Min/Max: {df_coords['Longitude'].min()} / {df_coords['Longitude'].max()}")

# Assess relational integrity prior to modeling integrations
print("\n======================================================")
print("🔗 [LEVEL 6] REFERENTIAL INTEGRITY CHECKS")
print("======================================================")

# 6A: Do the products in our shipments exist in our mapping table?
shipment_products = set(df_shipments['Product Name'].unique())
mapped_products = set(df_mapping['Product Name'].unique())

unmapped_products = shipment_products - mapped_products
print(f"Products in Shipments missing from Mapping: {len(unmapped_products)}")
if len(unmapped_products) > 0:
    print(f"Examples of missing products: {list(unmapped_products)[:3]}")

# 6B: Do the factories in our mapping table exist in our coordinate table?
mapped_factories = set(df_mapping['Factory'].unique())
coord_factories = set(df_coords['Factory'].unique())

missing_coords = mapped_factories - coord_factories
print(f"Factories missing coordinates: {len(missing_coords)}")
if len(missing_coords) > 0:
    print(f"Examples of missing factories: {list(missing_coords)[:3]}")
    
print("======================================================\n")

📊 DATASET PROFILE: MAIN SHIPMENTS

[LEVEL 1] SHAPE & SIZE:
Rows: 10,194
Columns: 18

[LEVEL 2 & 3] SCHEMA & COMPLETENESS:
               Data Type  Total Nulls  % Null  Unique Values
Row ID             int64            0     0.0          10194
Order ID          object            0     0.0           8549
Order Date        object            0     0.0            717
Ship Date         object            0     0.0           1338
Ship Mode         object            0     0.0              4
Customer ID        int64            0     0.0           5044
Country/Region    object            0     0.0              2
City              object            0     0.0            542
State/Province    object            0     0.0             59
Postal Code       object            0     0.0            654
Division          object            0     0.0              3
Region            object            0     0.0              4
Product ID        object            0     0.0             15
Product Name      object

In [4]:
# 3. DATA CLEANING

# Standardize string encoding to resolve entity resolution discrepancies
def normalize_names(df, col):
    return (df[col]
            .astype(str)
            .str.strip().str.replace(r'[–—−]', '-', regex=True)
            .str.replace(r'\s+', ' ', regex=True)
            .str.replace(r'\s*-\s*', ' - ', regex=True)
            .str.strip())

df_shipments['Product Name'] = normalize_names(df_shipments, 'Product Name')
df_mapping['Product Name'] = normalize_names(df_mapping, 'Product Name')

# Re-validate referential integrity post-normalization
shipment_products = set(df_shipments['Product Name'].unique())
mapped_products = set(df_mapping['Product Name'].unique())
print(f"Missing after normalization: {shipment_products - mapped_products}")

Missing after normalization: set()


In [5]:
# Standardize temporal features for downstream timedelta calculations
date_cols = ['Order Date', 'Ship Date']

for col in date_cols:
    df_shipments[col] = pd.to_datetime(df_shipments[col], format='%d-%m-%Y', errors='coerce')

In [6]:
# Isolate logical anomalies (negative shipping durations)
df_shipments['shipping_days'] = (
    df_shipments['Ship Date'] - df_shipments['Order Date']
).dt.days

invalid_ship_dates = df_shipments[df_shipments['shipping_days'] < 0]

In [7]:
# Remove surrogate keys with no analytical value
df_shipments = df_shipments.drop(columns=['Row ID'])

In [8]:
df_coords.columns

Index(['Factory', 'Latitude', 'Longitude'], dtype='object')

In [9]:
# Enforce standard snake_case naming conventions across all dataframes
rename_columns = {
    'Order ID': 'order_id',
    'Order Date': 'order_date',
    'Ship Date': 'ship_date',
    'Ship Mode': 'ship_mode',
    'Customer ID': 'customer_id',
    'Country/Region': 'country_region',
    'City': 'city',
    'State/Province': 'state_province',
    'Postal Code': 'postal_code',
    'Division': 'division',
    'Region': 'region',
    'Product ID': 'product_id',
    'Product Name': 'product_name',
    'Sales': 'sales',
    'Units': 'units',
    'Gross Profit': 'gross_profit',
    'Cost': 'cost',
    'shipping_days': 'shipping_days'
}

df_shipments.rename(columns=rename_columns, inplace=True)

In [10]:
rename_columns2 = {
    'Division': 'division',
    'Product Name': 'product_name',
    'Factory': 'factory'
}

df_mapping.rename(columns=rename_columns2, inplace=True)

In [11]:
rename_columns3 = {
    'Factory': 'factory',
    'Latitude': 'latitude',
    'Longitude': 'longitude'
}

df_coords.rename(columns=rename_columns3, inplace=True)

In [12]:
df_shipments['postal_code'].dtype

dtype('O')

In [13]:
# Export cleaned datasets to the interim staging layer
df_shipments.to_parquet('clean_shipments.parquet', index=False)
df_mapping.to_parquet('clean_mapping.parquet', index=False)
df_coords.to_parquet('clean_coords.parquet', index=False)